# Hello World — IBM Quantum

End-to-end walkthrough: authenticate to IBM Quantum, build a Bell state, run locally on Aer, then (optionally) on a real QPU.

**Security note — read this once.** Never paste your API token into a notebook cell. This notebook reads it from the `IBM_QUANTUM_TOKEN` environment variable. Set it before launching Jupyter:

- cmd: `set IBM_QUANTUM_TOKEN=...`
- PowerShell: `$env:IBM_QUANTUM_TOKEN = "..."`

The default flow below keeps the token **in memory only** — it lives inside the `service` object in this kernel session and is discarded when the kernel restarts. Nothing is written to disk. If you'd prefer qiskit's session-persistence (token saved to `~/.qiskit/qiskit-ibm.json` so you don't have to re-set the env var each session), there's an optional cell for that with the trade-offs spelled out.

In [ ]:
import sys, qiskit, qiskit_ibm_runtime
print('python  :', sys.executable)
print('qiskit  :', qiskit.__version__)
print('runtime :', qiskit_ibm_runtime.__version__)

## 1. Authenticate to IBM Quantum (in-memory)

Reads your token from the `IBM_QUANTUM_TOKEN` environment variable and builds a `service` object that holds it for this kernel session only. **Nothing is written to disk in this cell.**

Channel: `ibm_quantum_platform` (the unified channel for `quantum.cloud.ibm.com`).

In [ ]:
import os
from qiskit_ibm_runtime import QiskitRuntimeService

token = os.environ.get('IBM_QUANTUM_TOKEN')
if not token:
    raise RuntimeError(
        'Set IBM_QUANTUM_TOKEN in your shell before launching Jupyter.\n'
        '  cmd:        set IBM_QUANTUM_TOKEN=...\n'
        '  PowerShell: $env:IBM_QUANTUM_TOKEN = "..."'
    )

# In-memory authentication — the token is consumed by QiskitRuntimeService
# and held inside the `service` object for this kernel session only.
# We `del token` to remove the notebook-global reference so later cells
# (and `globals()` inspections) don't expose it. Jupyter cells share a
# kernel-level global namespace, so without `del`, the bare string would
# persist as a global between cells.
service = QiskitRuntimeService(channel='ibm_quantum_platform', token=token)
del token

print('authenticated (in-memory)')

### Optional: persist credentials to disk

If you'd rather not set `IBM_QUANTUM_TOKEN` every session, qiskit can save your account at `~/.qiskit/qiskit-ibm.json`. After that runs once, future notebooks (and Python sessions) can call `QiskitRuntimeService()` with no arguments and they'll find the saved token.

**Trade-off — be aware before running.** This writes a credential file to your home directory. The repo's `.gitignore` excludes it from accidental commits, but it remains on disk until you call `QiskitRuntimeService.delete_account()`. On shared machines or CI runners this is undesirable; on a personal laptop it's typically fine.

```python
# Uncomment to persist — only run this once per machine.
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel='ibm_quantum_platform',
#     token=os.environ['IBM_QUANTUM_TOKEN'],
#     overwrite=True,
# )
# print('account saved to ~/.qiskit/qiskit-ibm.json')
```

## 2. Inspect available backends

Reuse the `service` object from Section 1 — it's already authenticated. Listing backends shows which IBM Quantum systems your plan can submit to.

In [ ]:
backends = service.backends()
print(f'{len(backends)} backends visible to this account:')
for b in backends:
    print(f'  {b.name:20s}  qubits={b.num_qubits:3d}  '
          f'queued={b.status().pending_jobs}')

## 3. Build the Bell-state circuit

Two qubits: Hadamard on q0, then CNOT q0→q1. Measures should be perfectly correlated — `00` and `11` each ~50%, never `01` or `10`.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()
qc.draw('mpl')

## 4. Run locally on Aer (instant)

Always do this first — verifies your circuit works before spending QPU time. `AerSimulator` runs noiseless on your CPU; 1024 shots completes in milliseconds.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler

sim_backend = AerSimulator()
sampler = Sampler(mode=sim_backend)
job = sampler.run([qc], shots=1024)
counts = job.result()[0].data.meas.get_counts()
print('local Aer counts:', counts)

## 5. Run on real QPU hardware (optional — may queue)

Picks the least-busy operational backend, transpiles the circuit to its native gate set, and submits. The Open plan has limited monthly minutes and queue times vary from seconds to hours.

Skip this cell if you're just testing the setup.

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = service.least_busy(simulator=False, operational=True)
print(f'submitting to: {backend.name}  (queued jobs: {backend.status().pending_jobs})')

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
qc_isa = pm.run(qc)

sampler = Sampler(mode=backend)
job = sampler.run([qc_isa], shots=1024)
print(f'job id: {job.job_id()}')
print('result (blocks until job completes):')
result = job.result()
print(result[0].data.meas.get_counts())